# Agency Actuals Profiling
Owner: Mia
Table: `pondolens.dev_bronze.agency_actuals`
Baseline: 3,554 rows

## Goal
Identify data issues and comparable records for Q2. Do not modify Bronze.

## Checks
- Check counts by file/year, column types, nulls/blanks, and `_rescued_data`.
- Define what one row represents; check exact duplicates and repeated business keys.
- Identify agencies, parent departments, row types, and totals/detail records.
- Verify budget stages, expense classes, amount units, and numeric values.
- Check percentage types and whether values use a 0–100 or 0–1 scale.
- Identify matching allotments and obligations by agency, year, period, and expense class.
- Flag missing counterparts and zero denominators.
- Compare sample amounts and calculated rates with the original source.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| To be completed | | |

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| To be completed | | Pending |

## Semantic Notes
- Unobligated allotments = matching allotments − obligations.
- Obligation rate = obligations ÷ allotments × 100.
- Missing/zero denominators: flag and return null.
- Overall rate: divide summed obligations by summed allotments; do not average rates.
- Document units and avoid combining totals with expense-class details.

## Conclusion
- Required Silver changes:
- Ready for Q2? Why / why not:
- Open questions:

In [0]:
%sql
-- Agency Actuals profiling (Databricks SQL). READ-ONLY on Bronze: only TEMP views are created.
-- Table: pondolens.dev_bronze.agency_actuals | Baseline: 3,554 rows | Units: amounts = thousand pesos, rates = 0-100
-- Run cell by cell. Tolerance for reconciliations = 1 (thousand pesos).

-- 0. Clean view: real NULLs, parsed numbers ('-' -> 0, '(x)' -> -x, '12.3%' -> 12.3), typed keys
CREATE OR REPLACE TEMP VIEW v AS
SELECT *, CAST(year AS INT) AS yr, CAST(source_row AS INT) AS src_row,
  NULLIF(NULLIF(expense_class,'null'),'') AS ec,
  CASE WHEN amount IS NULL OR lower(trim(amount)) IN ('null','') THEN NULL
       WHEN trim(amount) = '-' THEN 0
       WHEN amount LIKE '(%)' THEN -try_cast(translate(amount,'(),','') AS DOUBLE)
       ELSE try_cast(translate(amount,',','') AS DOUBLE) END AS amt,
  try_cast(replace(percent_amount,'%','') AS DOUBLE) AS pct
FROM pondolens.dev_bronze.agency_actuals;

-- 1. Counts by file/year vs baseline, and column types
SELECT _source_file, source_name, source_page, year, count(*) AS n, sum(count(*)) OVER () AS total, 3554 AS baseline
FROM v GROUP BY 1,2,3,4;
DESCRIBE TABLE pondolens.dev_bronze.agency_actuals;

-- 2. Nulls / blanks / 'null' text / dashes per column (assumes string columns; CAST to STRING if typed)
SELECT col, count(*) AS n,
  sum(CASE WHEN val IS NULL THEN 1 ELSE 0 END) AS sql_null,
  sum(CASE WHEN lower(trim(val)) = 'null' THEN 1 ELSE 0 END) AS null_text,
  sum(CASE WHEN trim(val) = '' THEN 1 ELSE 0 END) AS blank,
  sum(CASE WHEN trim(val) = '-' THEN 1 ELSE 0 END) AS dash,
  count(DISTINCT val) AS distinct_vals
FROM (SELECT stack(16,'source_name',source_name,'source_page',source_page,'source_row',source_row,'year',year,
        'category',category,'particulars',particulars,'row_type',row_type,'budget_stage',budget_stage,
        'expense_class',expense_class,'unit_amount',unit_amount,'amount',amount,'percent_type',percent_type,
        'percent_amount',percent_amount,'_rescued_data',_rescued_data,'_source_file',_source_file,
        '_ingested_at',CAST(_ingested_at AS STRING)) AS (col, val) FROM pondolens.dev_bronze.agency_actuals)
GROUP BY col;

-- 3. Grain: one row = one measure (budget_stage x expense_class, or one percent) of one source line in one year.
SELECT 'rows with _rescued_data' AS chk, count(*) AS n FROM v WHERE _rescued_data IS NOT NULL AND lower(_rescued_data) <> 'null'
UNION ALL SELECT 'exact duplicate rows', count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM pondolens.dev_bronze.agency_actuals)) FROM v
UNION ALL SELECT 'repeated business keys (incl. category)', count(*) FROM
  (SELECT 1 FROM v GROUP BY yr, category, particulars, row_type, budget_stage, ec, percent_type HAVING count(*) > 1)
UNION ALL SELECT 'repeated keys if category ignored', count(*) FROM
  (SELECT 1 FROM v GROUP BY yr, particulars, budget_stage, ec, percent_type HAVING count(*) > 1);
SELECT n AS records_per_source_line, count(*) AS lines   -- 35 = full set; fewer = lines missing Disbursements/rates
FROM (SELECT yr, src_row, count(*) AS n FROM v GROUP BY 1,2) GROUP BY n;

-- 4. Entities: row types, agencies that exist in only one year (renames), stages x classes x units
SELECT category, row_type, count(DISTINCT yr * 1000 + src_row) AS source_lines FROM v GROUP BY 1,2 ORDER BY 1,2;
SELECT particulars, min(yr) AS only_in_year FROM v WHERE row_type = 'agency_total'
GROUP BY particulars HAVING count(DISTINCT yr) = 1;
SELECT budget_stage, ec, unit_amount, count(*) AS n FROM v GROUP BY 1,2,3 ORDER BY 1,2;

-- 5. Totals vs detail (ec = 'Total'): agencies/line items vs category total, categories vs Grand Total
SELECT yr, category, budget_stage,
  round(sum(CASE WHEN row_type = 'agency_total' THEN amt END) - sum(CASE WHEN row_type = 'category_total' THEN amt END), 2) AS agencies_minus_total,
  round(sum(CASE WHEN row_type = 'line_item' THEN amt END) - sum(CASE WHEN row_type = 'category_total' THEN amt END), 2) AS lines_minus_total
FROM v WHERE ec = 'Total' AND category <> 'Grand Total' GROUP BY 1,2,3 ORDER BY 1,2,3;
SELECT yr, budget_stage,
  round(sum(CASE WHEN category <> 'Grand Total' AND row_type = 'category_total' THEN amt END) - sum(CASE WHEN category = 'Grand Total' THEN amt END), 2) AS categories_minus_grand
FROM v WHERE ec = 'Total' GROUP BY 1,2 ORDER BY 1,2;

-- 6. Amount formats, negatives, and arithmetic identities (view `chk` = violations only)
SELECT CASE WHEN amount IS NULL OR lower(trim(amount)) = 'null' THEN 'null' WHEN trim(amount) = '-' THEN 'dash = 0'
            WHEN amount LIKE '(%)' THEN 'parentheses = negative' WHEN amount LIKE '%,%' THEN 'thousands separator'
            WHEN amt IS NULL THEN 'UNPARSEABLE' ELSE 'plain number' END AS amount_format, count(*) AS n,
       sum(CASE WHEN amt < 0 THEN 1 ELSE 0 END) AS negatives
FROM v WHERE unit_amount <> 'percent' OR amount NOT IN ('null') GROUP BY 1;

CREATE OR REPLACE TEMP VIEW chk AS
SELECT * FROM (
  SELECT 'PS+MOOE+FINEX+CO <> Total' AS chk, yr, src_row, budget_stage AS item,
    sum(CASE WHEN ec <> 'Total' THEN amt END) - sum(CASE WHEN ec = 'Total' THEN amt END) AS diff
  FROM v WHERE ec IS NOT NULL GROUP BY 2,3,4
  UNION ALL
  SELECT 'Appropriations + Adjustments <> Total Available', yr, src_row, ec,
    sum(CASE WHEN budget_stage IN ('Appropriations','Adjustments') THEN amt END)
      - sum(CASE WHEN budget_stage = 'Total Available Appropriations' THEN amt END)
  FROM v WHERE ec IS NOT NULL GROUP BY 2,3,4
  UNION ALL
  SELECT 'Allotments - Obligations <> Unobligated', yr, src_row, ec,
    sum(CASE WHEN budget_stage = 'Allotments' THEN amt END) - sum(CASE WHEN budget_stage = 'Obligations Incurred' THEN amt END)
      - sum(CASE WHEN budget_stage = 'Unobligated Allotments' THEN amt END)
  FROM v WHERE ec IS NOT NULL GROUP BY 2,3,4
  UNION ALL
  SELECT 'Total Available - Unreleased <> Allotments', yr, src_row, 'Total',
    sum(CASE WHEN budget_stage = 'Total Available Appropriations' THEN amt END)
      - sum(CASE WHEN budget_stage = 'Unreleased Appropriations' THEN amt END)
      - sum(CASE WHEN budget_stage = 'Allotments' THEN amt END)
  FROM v WHERE ec = 'Total' OR budget_stage = 'Unreleased Appropriations' GROUP BY 2,3
) WHERE abs(diff) > 1;
SELECT c.*, p.particulars FROM chk c JOIN (SELECT DISTINCT yr, src_row, particulars FROM v) p USING (yr, src_row);

-- 7. Percentages: format, scale (0-100 vs 0-1), Excel errors
SELECT percent_type, count(*) AS n,
  sum(CASE WHEN substr(percent_amount,-1) = '%' THEN 1 ELSE 0 END) AS with_pct_suffix,
  sum(CASE WHEN percent_amount LIKE '#%' THEN 1 ELSE 0 END) AS excel_errors,
  min(pct) AS min_pct, max(pct) AS max_pct,
  sum(CASE WHEN pct <= 1 THEN 1 ELSE 0 END) AS le_1, sum(CASE WHEN pct > 100 THEN 1 ELSE 0 END) AS gt_100
FROM v WHERE unit_amount = 'percent' GROUP BY 1;

-- 8. One row per source line: totals + published rates; recompute rates and test which denominator they use
CREATE OR REPLACE TEMP VIEW r AS
SELECT yr, src_row, max(particulars) AS particulars, max(row_type) AS row_type,
  sum(CASE WHEN budget_stage = 'Allotments' AND ec = 'Total' THEN amt END) AS allot,
  sum(CASE WHEN budget_stage = 'Obligations Incurred' AND ec = 'Total' THEN amt END) AS obl,
  sum(CASE WHEN budget_stage = 'Appropriations' AND ec = 'Total' THEN amt END) AS appr,
  sum(CASE WHEN budget_stage = 'Total Available Appropriations' AND ec = 'Total' THEN amt END) AS taa,
  sum(CASE WHEN budget_stage = 'Disbursements' AND unit_amount <> 'percent' THEN amt END) AS disb,
  max(CASE WHEN percent_type = 'Obligation rates' THEN pct END) AS pub_obl,
  max(CASE WHEN percent_type = 'disbursement_rates_obligations' THEN pct END) AS pub_disb_obl,
  max(CASE WHEN percent_type = 'disbursement_rates_appropriations' THEN pct END) AS pub_disb_appr
FROM v GROUP BY 1,2;
SELECT count(pub_obl) AS rates_checked,
  max(abs(pub_obl      - obl  * 100.0 / NULLIF(allot,0))) AS obligation_rate_maxdiff,
  max(abs(pub_disb_obl - disb * 100.0 / NULLIF(obl,0)))   AS disb_vs_obligations_maxdiff,
  max(abs(pub_disb_appr - disb * 100.0 / NULLIF(appr,0))) AS disb_appr_if_Appropriations,
  max(abs(pub_disb_appr - disb * 100.0 / NULLIF(taa,0)))  AS disb_appr_if_TotalAvailable
FROM r;   -- diffs <= 0.05 mean the denominator matches (published rates are rounded to 0.1)

-- 9. Allotment <-> obligation matching by year x source line x expense class (period = fiscal year; no period column)
CREATE OR REPLACE TEMP VIEW m AS
SELECT yr, src_row, max(particulars) AS particulars, max(row_type) AS row_type, ec,
  sum(CASE WHEN budget_stage = 'Allotments' THEN amt END) AS allot,
  sum(CASE WHEN budget_stage = 'Obligations Incurred' THEN amt END) AS obl,
  sum(CASE WHEN budget_stage = 'Allotments' THEN amt END) - sum(CASE WHEN budget_stage = 'Obligations Incurred' THEN amt END) AS unobligated,
  sum(CASE WHEN budget_stage = 'Obligations Incurred' THEN amt END) * 100.0
    / NULLIF(sum(CASE WHEN budget_stage = 'Allotments' THEN amt END), 0) AS obligation_rate   -- NULL when allotment is missing/zero
FROM v WHERE ec IS NOT NULL AND budget_stage IN ('Allotments','Obligations Incurred') GROUP BY yr, src_row, ec;
SELECT ec, CASE WHEN allot IS NULL OR obl IS NULL THEN 'MISSING_COUNTERPART' WHEN allot = 0 AND obl = 0 THEN 'BOTH_ZERO (rate NULL)'
                WHEN allot = 0 THEN 'ZERO_ALLOT_NONZERO_OBL' WHEN obl = 0 THEN 'ALLOT_NO_OBLIGATION'
                WHEN obl > allot + 1 THEN 'OBL_EXCEEDS_ALLOT' ELSE 'OK' END AS status, count(*) AS n
FROM m GROUP BY 1,2 ORDER BY 1,2;
SELECT yr, particulars FROM m WHERE ec = 'Total' AND (allot IS NULL OR allot = 0) ORDER BY 1,2;   -- zero denominators (lines)
-- Overall rate = SUM/SUM (never the average of rates); NGA agencies only, Total class
SELECT yr, sum(obl) * 100.0 / NULLIF(sum(allot),0) AS overall_obligation_rate, avg(obligation_rate) AS wrong_avg_of_rates
FROM m WHERE row_type = 'agency_total' AND ec = 'Total' GROUP BY 1;

-- 10. Sample for comparison with the original workbook (source_row = Excel row on sheet `source_page`)
SELECT * FROM (SELECT source_name, source_page, source_row, yr, particulars, budget_stage, ec, amount, unit_amount,
  row_number() OVER (PARTITION BY yr, budget_stage ORDER BY md5(concat_ws('|', yr, src_row, budget_stage, ec))) AS rn
  FROM v WHERE unit_amount <> 'percent' AND row_type IN ('agency_total','category_total')) WHERE rn <= 3;

-- 11. Findings register: live evidence + proposed Silver treatment
SELECT * FROM (
  SELECT 1 AS id, 'Row count vs baseline' AS issue, count(*) AS evidence, 'Must equal 3,554' AS silver_treatment FROM v
  UNION ALL SELECT 2, '_rescued_data populated', count(*), 'Drop column if 0, else quarantine' FROM v WHERE _rescued_data IS NOT NULL AND lower(_rescued_data) <> 'null'
  UNION ALL SELECT 3, 'Missing values stored as text ''null''', count(*), 'Convert to real NULL' FROM v WHERE 'null' IN (expense_class, amount, percent_type, percent_amount, particulars)
  UNION ALL SELECT 4, 'Amounts as text: ''-'' or (negative)', count(*), 'Parse: - -> 0, (x) -> -x, strip commas, DECIMAL' FROM v WHERE trim(amount) = '-' OR amount LIKE '(%)'
  UNION ALL SELECT 5, 'Percent as text with % (0-100 scale)', count(*), 'Strip %, keep 0-100, document scale' FROM v WHERE substr(percent_amount,-1) = '%'
  UNION ALL SELECT 6, '#DIV/0! in percent_amount', count(*), 'NULL + zero_denominator flag' FROM v WHERE percent_amount LIKE '#%'
  UNION ALL SELECT 7, 'expense_class NULL on amount rows (Disbursements/Unreleased)', count(*), 'Explicit ALL/N-A, exclude from by-class analysis' FROM v WHERE ec IS NULL AND unit_amount <> 'percent'
  UNION ALL SELECT 8, 'Total/summary rows mixed with detail', count(*), 'is_total + is_class_total flags, never sum together' FROM v WHERE row_type <> 'agency_total' OR ec = 'Total'
  UNION ALL SELECT 9, 'Agencies present in one year only (rename)', count(*), 'dim_agency mapping' FROM (SELECT particulars FROM v WHERE row_type = 'agency_total' GROUP BY 1 HAVING count(DISTINCT yr) = 1)
  UNION ALL SELECT 10, 'Identity violations (see view chk)', count(*), 'Flag rows, use Totals only if they reconcile' FROM chk
  UNION ALL SELECT 11, 'Source lines with < 35 records (no Disbursements/rates)', count(*), 'Keep, NULL not zero' FROM (SELECT 1 FROM v GROUP BY yr, src_row HAVING count(*) < 35)
  UNION ALL SELECT 12, 'Zero/missing allotment lines (Total class)', count(*), 'Rate NULL + flag' FROM m WHERE ec = 'Total' AND (allot IS NULL OR allot = 0)
  UNION ALL SELECT 13, 'disbursement_rates_appropriations off by >0.051 vs Appropriations', count(*), 'If >0 and 0 vs Total Available: rename column' FROM r WHERE abs(pub_disb_appr - disb * 100.0 / NULLIF(appr,0)) > 0.051
  UNION ALL SELECT 14, 'Inconsistent source_page labels', count(DISTINCT source_page), 'Standardise or drop (>1 = inconsistent)' FROM v
) ORDER BY id;

-- Verdict (from the Python run on the same file): ready for Q2 AFTER the Silver treatments above. Open questions:
-- FY2025 cut-off date; Office of the President FY2024 MOOE appropriation (class sum <> Total); NEDA -> DEPDev as one agency; parent-department mapping.